# TweetAge · 02 Experimentos
l
Este notebook reúne los experimentos que se reaalizarno por el equipo para la solución del reto.

**Regla de la competencia:** solo modelos de lenguaje propios entrenados con los datos del reto (n-gramas de caracteres y palabras). Nada de vectorización (BoW, TF-IDF), embeddings, transformers ni modelos preentrenados.

**Estructura**

1. Datos y estrategia de validación.
2. Canales: cada modelo de lenguaje que aporta probabilidades.
3. Construcción incremental de la mezcla, canal por canal.
4. Experimento: orden del LM secuencial de palabras (2 vs 3).
5. Resultado por clase de la configuración final.
6. Registro de experimentos.

Los modelos y la mezcla viven en el paquete `tweetage` (`src/tweetage/`); aquí solo se usan. Este notebook no genera submission.

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

from tweetage import cache, paths, registry, validation
from tweetage import models as M
from tweetage import pipeline as P

CLASES = P.CLASES
SEED_AJUSTE = 20261108  # se usa para elegir hiperparámetros
SEED_CONFIRMACION = 20261109  # partición independiente para confirmar la decisión
SEEDS = (SEED_AJUSTE, SEED_CONFIRMACION)

## 1. Datos y estrategia de validación

Hay textos repetidos en el train (el mismo tweet, a veces con otra etiqueta). Si un texto quedara a la vez en entrenamiento y validación, el modelo lo "recordaría" y la métrica saldría inflada. Por eso la partición se hace con **`StratifiedGroupKFold`**:

- **Grupo** = el texto normalizado (sin espacios en los extremos y en minúsculas): todas las copias de un texto caen del mismo lado.
- **Estratificado** por rango de edad, para mantener la proporción de clases.
- Se toma uno de 5 folds como validación (≈20 %).

La partición está en `tweetage.validation`, para que el análisis de errores (notebook 03) use exactamente la misma.

Se usan **dos semillas**. Con la de *ajuste* se eligen hiperparámetros; con la de *confirmación* se comprueba la decisión en datos que no se usaron para elegir. Un cambio solo se acepta si mejora en ambas.

In [ ]:
train, test = paths.load_data()
X = paths.texts(train)
y = train["age_range"].to_numpy()
print(
    f"Filas train: {len(X):,} | textos únicos: {len(set(validation.groups_of(X))):,} | filas test: {len(test):,}"
)
display(train["age_range"].value_counts().reindex(CLASES).to_frame("tweets"))

In [ ]:
# validation.split verifica que ningún texto quede en ambos lados y cachea los índices
splits = {seed: validation.split(X, y, seed) for seed in SEEDS}
for seed, (Xtr, _, Xva, _) in splits.items():
    print(f"Seed {seed}: train={len(Xtr):,} valid={len(Xva):,} textos compartidos=0")

## 2. Canales

Cada canal es un modelo de lenguaje **condicional a la clase**: estima qué tan probable es el texto si lo escribió alguien de cada rango de edad, y lo convierte en probabilidades con una softmax (la temperatura controla qué tan "seguras" son).

| Canal | Modelo | Qué captura | α | Temp. |
|---|---|---|---|---|
| `word1` | Naive Bayes multinomial, unigramas de palabras | vocabulario | 0.01 | 1 |
| `char4`, `char5`, `char6` | Naive Bayes multinomial, n-gramas de caracteres | ortografía, abreviaturas, emojis, alargamientos | 0.1 / 0.1 / 0.03 | 1 |
| `hashtags` | categórico sobre hashtags únicos | temas | 0.1 | 1 |
| `mentions` | categórico sobre menciones únicas | a quién le hablan | 0.1 | 0.35 (en la mezcla) |
| `position` | multinomial sobre (zona, token): primer y último cuarto del tweet | cómo empiezan y terminan los mensajes | 0.03 | 1 |
| `mention_pos` | categórico sobre (mención inicial / dentro del texto, usuario) | respuestas vs. menciones | 0.01 | 1 |
| `char_seq2` | LM de Markov de caracteres, contexto de 2 caracteres | secuencia de caracteres | 0.1 | 0.05 |
| `word_seq2/3` | LM de Markov de palabras, contexto de 1 o 2 palabras | secuencia de palabras | 0.1 | grilla |

Las configuraciones están en `pipeline.CHANNELS`. Las probabilidades de cada canal se guardan en caché: la primera ejecución entrena todo; las siguientes cargan del disco.

Para el experimento de la sección 4 los LMs de palabras se guardan a temperatura 1; luego se re-escalan a cualquier temperatura sin reentrenar, porque softmax(log p / T) = softmax(puntaje / T).

In [ ]:
# Canales extra para la sección 4: LMs de palabras de orden 2 y 3 a temperatura 1.
CHANNELS = {
    **P.CHANNELS,
    "word_seq2_t1": (M.WordSequenceLM, {"alpha": 0.1, "min_count": 3, "order": 2}, 1.0),
    "word_seq3_t1": (M.WordSequenceLM, {"alpha": 0.1, "min_count": 3, "order": 3}, 1.0),
}

probs, masks = {}, {}
for seed, (Xtr, ytr, Xva, _) in splits.items():
    t0 = time.perf_counter()
    probs[seed] = {
        name: cache.channel(name, Xtr, ytr, Xva, tag=f"seed{seed}", channels=CHANNELS) for name in CHANNELS
    }
    masks[seed] = M.PositionAwareMentionLM.masks(Xva)["all"]
    print(f"Seed {seed}: {len(CHANNELS)} canales en {time.perf_counter() - t0:.1f} s")

In [ ]:
def evaluar(prob, y_true):
    pred = np.asarray(CLASES)[prob.argmax(axis=1)]
    return f1_score(y_true, pred, average="macro"), accuracy_score(y_true, pred)


def con_temperatura(p_t1, temperature):
    # p_t1 = softmax(puntaje)  =>  softmax(log(p_t1) / T) = softmax(puntaje / T)
    return M.softmax(np.log(np.clip(p_t1, 1e-300, 1.0)) / temperature)


# F1 de cada canal por separado, como referencia
filas = []
for name in P.CHANNELS:
    f1s = [evaluar(probs[s][name], splits[s][3])[0] for s in SEEDS]
    filas.append({"canal": name, "f1_ajuste": f1s[0], "f1_confirmacion": f1s[1]})
pd.DataFrame(filas).set_index("canal").round(4)

## 3. Construcción incremental de la mezcla

La mezcla se armó agregando un canal a la vez, como promedio ponderado de probabilidades. Cada paso se conservó solo si mejoraba en validación y luego en Kaggle. Los pesos son los ya validados (`pipeline.blend_steps`):

1. **`char_mix`**: palabras + caracteres 4/5/6 (pesos 0.216 / 0.144 / 0.240 / 0.400).
2. **`hashtags`**: se agrega el canal de hashtags (0.36 · char_mix + 0.40 · hashtags, renormalizado).
3. **`mentions`**: 70 % lo anterior + 30 % menciones, con temperatura 0.35.
4. **`position`**: 90 % + 10 % canal posicional.
5. **`mention_pos`**: solo en tweets con al menos una mención, 70 % + 30 % menciones con posición.
6. **`char_seq2`**: 85 % + 15 % LM secuencial de caracteres.
7. **`word_seq3`**: 80 % + 20 % LM secuencial de palabras de orden 3 (ver sección 4).

Las celdas siguientes recalculan cada paso en las dos semillas y lo registran.

In [ ]:
KAGGLE = {"position": 0.58850, "char_seq2": 0.59281, "word_seq3": 0.60021}
DESCRIPCION = {
    "char_mix": "word1 + char4/5/6",
    "hashtags": "+ hashtags",
    "mentions": "+ menciones (T=0.35), peso 0.30",
    "position": "+ posición start_end, peso 0.10",
    "mention_pos": "+ menciones con posición, peso 0.30 solo en tweets con mención",
    "char_seq2": "+ LM secuencial de caracteres orden 2 (T=0.05), peso 0.15",
    "word_seq3": "+ LM secuencial de palabras orden 3 (T=0.2), peso 0.20",
}

pasos = {s: P.blend_steps(probs[s], masks[s]) for s in SEEDS}
filas = []
for i, step in enumerate(pasos[SEED_AJUSTE], start=1):
    nombre = f"mezcla_{i}_{step}"
    fila = {"paso": nombre, "descripcion": DESCRIPCION[step], "kaggle": KAGGLE.get(step)}
    for s, col in zip(SEEDS, ("ajuste", "confirmacion"), strict=True):
        f1, acc = evaluar(pasos[s][step], splits[s][3])
        fila[f"f1_{col}"] = f1
        registry.log(
            nombre,
            {"paso": step, "descripcion": DESCRIPCION[step]},
            s,
            f1,
            acc,
            notas="mezcla incremental",
            kaggle_public=KAGGLE.get(step),
        )
    filas.append(fila)

tabla_mezcla = pd.DataFrame(filas).set_index("paso")
tabla_mezcla["delta_ajuste"] = tabla_mezcla["f1_ajuste"].diff()
tabla_mezcla["delta_confirmacion"] = tabla_mezcla["f1_confirmacion"].diff()
tabla_mezcla.round({c: 4 for c in tabla_mezcla.columns if c.startswith(("f1", "delta"))})

Las mejoras locales por paso son pequeñas (milésimas de F1), pero en el test de Kaggle cada paso confirmado sumó más que eso. Por eso se exige que la mejora sea consistente en las dos semillas antes de gastar un envío.

## 4. Experimento: orden del LM secuencial de palabras

**Pregunta:** ¿conviene un LM de palabras con contexto de una palabra (orden 2) o de dos (orden 3)?

**Base:** la mezcla hasta `char_seq2`, sin ningún LM de palabras.

**Procedimiento**

1. En la semilla de ajuste, para cada orden se prueba una grilla de temperatura × peso: mezcla = (1 − peso) · base + peso · LM.
2. Se elige la mejor configuración **de cada orden** por F1 macro.
3. Ambas se evalúan en la semilla de confirmación y se comparan contra la base y entre sí.

In [ ]:
TEMPERATURAS = (0.05, 0.1, 0.2, 0.3)
PESOS = (0.05, 0.10, 0.15, 0.20)


def mezcla_palabras(seed, orden, temperatura, peso):
    base = pasos[seed]["char_seq2"]
    lm = con_temperatura(probs[seed][f"word_seq{orden}_t1"], temperatura)
    return (1 - peso) * base + peso * lm


yva = splits[SEED_AJUSTE][3]
filas = [
    {
        "orden": None,
        "temperatura": None,
        "peso": 0.0,
        "f1_macro": evaluar(pasos[SEED_AJUSTE]["char_seq2"], yva)[0],
    }
]
for orden in (2, 3):
    for t in TEMPERATURAS:
        for w in PESOS:
            f1, _ = evaluar(mezcla_palabras(SEED_AJUSTE, orden, t, w), yva)
            filas.append({"orden": orden, "temperatura": t, "peso": w, "f1_macro": f1})

grilla = pd.DataFrame(filas).sort_values("f1_macro", ascending=False).reset_index(drop=True)
grilla.head(12).round(4)

In [ ]:
# Mejor configuración de cada orden en la semilla de ajuste
mejores = {}
for orden in (2, 3):
    fila = grilla[grilla["orden"] == orden].iloc[0]
    mejores[orden] = (float(fila["temperatura"]), float(fila["peso"]))
    print(
        f"Orden {orden}: temperatura={mejores[orden][0]}, peso={mejores[orden][1]}, F1 ajuste={fila['f1_macro']:.4f}"
    )

In [ ]:
# Confirmación en la partición independiente
yva_c = splits[SEED_CONFIRMACION][3]
f1_base_c, acc_base_c = evaluar(pasos[SEED_CONFIRMACION]["char_seq2"], yva_c)
filas = [
    {
        "variante": "base (hasta char_seq2)",
        "f1_ajuste": grilla.query("peso == 0")["f1_macro"].iloc[0],
        "f1_confirmacion": f1_base_c,
    }
]

for orden, (t, w) in mejores.items():
    nombre = f"lm_palabras_orden{orden}"
    config = {
        "orden": orden,
        "temperatura": t,
        "peso": w,
        "alpha": 0.1,
        "min_count": 3,
        "base": "mezcla_6_char_seq2",
    }
    fila = {"variante": f"{nombre} (T={t}, peso={w})"}
    for s, col in zip(SEEDS, ("ajuste", "confirmacion"), strict=True):
        f1, acc = evaluar(mezcla_palabras(s, orden, t, w), splits[s][3])
        fila[f"f1_{col}"] = f1
        registry.log(nombre, config, s, f1, acc, notas="experimento orden LM palabras")
    filas.append(fila)

confirmacion = pd.DataFrame(filas).set_index("variante")
confirmacion["delta_vs_base_confirmacion"] = confirmacion["f1_confirmacion"] - f1_base_c
confirmacion.round(4)

**Decisión:** se adopta el LM de palabras de **orden 3** con la configuración elegida en la semilla de ajuste, porque supera al orden 2 y a la base en ambas particiones. Es la configuración de `pipeline.CHANNELS["word_seq3"]` y `pipeline.WORD_SEQ_WEIGHT`.

La celda siguiente comprueba que la mezcla final del paquete coincide con la del experimento.

In [ ]:
t3, w3 = mejores[3]
assert (t3, w3) == (P.CHANNELS["word_seq3"][2], P.WORD_SEQ_WEIGHT), (
    "La grilla eligió otra configuración que la del paquete"
)
for s in SEEDS:
    np.testing.assert_allclose(mezcla_palabras(s, 3, t3, w3), P.blend(probs[s], masks[s]), atol=1e-12)
print("La mezcla del paquete coincide con la elegida en el experimento.")

## 5. Resultado por clase de la configuración final

Evaluado en la semilla de confirmación. Las clases centrales (18-24, 25-34) son las más difíciles: sus estilos de escritura se parecen entre sí y a las clases vecinas.

In [ ]:
pred_final = np.asarray(CLASES)[P.blend(probs[SEED_CONFIRMACION], masks[SEED_CONFIRMACION]).argmax(axis=1)]
print(classification_report(yva_c, pred_final, labels=CLASES, digits=3, zero_division=0))

## 6. Registro de experimentos

Cada experimento queda con su configuración, semilla, F1, accuracy y, cuando existe, el score público de Kaggle. Para anotar el score de una submission nueva: `registry.set_kaggle_score(id, score)`.

In [ ]:
registry.show()

# Parte 2 · Tanda de experimentos v2

Experimentos motivados por el análisis exploratorio (`01_exploracion`). Las secciones 1–6 quedan **sin cambios**; aquí se reutilizan sus particiones, su caché y su mezcla vigente (0.60021 en Kaggle) como referencia.

**Índice**

| Sección | Contenido | Resultado |
|---|---|---|
| 7 | Referencia: mezcla vigente | reproduce 0.5849 / 0.5869 |
| 8 | Grupo A: preprocesamiento | solo el tokenizador ayuda, poco |
| 9 | Grupo B: pesos y sesgos de la mezcla | ninguno mejora |
| 10 | Grupo C: canales nuevos | mejoras pequeñas como canales agregados |
| 11 | Ranking de los grupos A–C | ninguno pasa la regla por sí solo |
| 12 | Combinación automática (mezcla conjunta) | descartada: Δ ≈ 0 |
| 13 | Combinación greedy (reemplazos + canales agregados) | **mejora clara** |
| 14 | Combinación con el LM de entidades del equipo | **mejor resultado** |
| 15 | Resumen global: experimentos, canales, familias y Kaggle | tablas para el informe |
| 16 | Conclusiones | |

**Protocolo** (el mismo de las secciones anteriores, con dos agregados):

1. Todo parámetro (pesos, temperaturas, sesgos, grillas) se elige **solo en la semilla de ajuste** (20261108).
2. Se evalúa en la **semilla de confirmación** (20261109), que no se usó para elegir.
3. Nuevo: **Δ pareado** frente a la mezcla vigente en ambas semillas e **intervalo bootstrap al 95 %** del Δ en confirmación (1 000 remuestreos de las mismas filas).
4. Nuevo: regla explícita. Un experimento **pasa** si Δ > 0 en ambas semillas y el límite inferior del intervalo es > 0. Es **fuerte** si además el Δ medio es ≥ 0.002.

**Grupos de experimentos**

| Grupo | ID | Qué cambia | ¿Reentrena? |
|---|---|---|---|
| A · preprocesamiento | A1 | `html.unescape` en todos los textos | sí, los 10 canales |
| | A2 | `TweetTokenizer` (NLTK, solo reglas) en word1, position, mention_pos y word_seq3 | sí, 4 canales |
| | A3 | A1 + A2 | sí, los 10 canales |
| | A4 | quitar de train textos vacíos y `#¿NOMBRE?` | sí, los 10 canales |
| | A5 | quitar de train plantillas N2 ruidosas (pureza ≤ 0.5, n ≥ 5, calculada solo con el train de cada semilla) | sí, los 10 canales |
| B · mezcla | B1 | sesgos por clase sobre la mezcla vigente | no |
| | B2 | pesos y temperaturas ajustados en conjunto (interpolación lineal) | no |
| | B3 | B2 + compuertas: menciones y hashtags solo votan si el tweet los tiene | no |
| | B4 | interpolación log-lineal | no |
| C · canales nuevos | C1 | word_seq3 con interpolación Jelinek-Mercer (3 → 2 → 1) | sí |
| | C2 | caracteres sin casefold (`char5_cs`) | sí |
| | C3 | LM de forma (A/a/9/puntuación), órdenes 5 y 7 | sí |
| | C4 | Markov en reversa (caracteres y palabras) y Markov de caracteres de orden 4 | sí |
| | C5 | longitud como evento categórico | sí |
| | C6 | especialistas 18-24 frente a 25-34 | sí |
| G · greedy | G1–G3 | reemplazos sin casefold + tokenizador + shape7 + char_seq4, paso a paso | sí |
| E · entidades | E0, E_* | LM de palabras que conserva menciones/hashtags frecuentes (0.60270), solo y combinado | sí |

**Código**: modelos nuevos en `tweetage.models_extra`, canales y variantes en `tweetage.channels_extra`, mezcla en `tweetage.blend_opt`, evaluación en `tweetage.evaluation` y orquestación en `tweetage.experiments_v2`. Las especificaciones que se guardan aquí (`experiments/spec_*.json`) las usa `04_submission_v2.ipynb` para entrenar con todo el train y generar el CSV.

**Bibliotecas externas nuevas**: NLTK (`TweetTokenizer`, solo expresiones regulares; Apache 2.0) y SciPy (`optimize`; BSD-3).

In [ ]:
from tweetage import blend_opt as B
from tweetage import channels_extra as CX
from tweetage import evaluation as E
from tweetage import experiments_v2 as XV

N_JOBS = 2  # procesos en paralelo (en el equipo del equipo se usó 6)
AJ, CF = SEED_AJUSTE, SEED_CONFIRMACION
COLS = [
    "f1_ajuste",
    "f1_confirmacion",
    "delta_ajuste",
    "delta_confirmacion",
    "ic95_bajo",
    "ic95_alto",
    "pred_cambiadas",
    "pasa",
    "fuerte",
]
resultados = []  # una fila por experimento

# Scores públicos de Kaggle conocidos de experimentos de la Parte 2 (los de la Parte 1 están en la sección 3)
KAGGLE_V2 = {"E0_ent5": 0.60270}  # submission del equipo: LM de palabras con entidades frecuentes (umbral 5)


def evaluar_v2(nombre, cand, config, notas=""):
    """Compara ``cand`` (dict semilla -> probabilidades o puntajes) contra la mezcla vigente."""
    fila = E.compare(
        nombre, base_runs[AJ].y_idx, p_base[AJ], cand[AJ], base_runs[CF].y_idx, p_base[CF], cand[CF]
    )
    resultados[:] = [r for r in resultados if r["experimento"] != nombre] + [
        {**fila, "grupo": config.get("grupo", "")}
    ]
    for s, col in ((AJ, "ajuste"), (CF, "confirmacion")):
        registry.log(
            f"v2_{nombre}",
            config,
            s,
            fila[f"f1_{col}"],
            fila[f"acc_{col}"],
            notas=notas,
            kaggle_public=KAGGLE_V2.get(nombre),
        )
    return fila


def tabla(grupo=None):
    df = pd.DataFrame(resultados).set_index("experimento")
    if grupo:
        df = df[df["grupo"] == grupo]
    return df[COLS].round(4)

## 7. Referencia: mezcla vigente

Se cargan de la caché los 10 canales de la sección 2 (mismos archivos, no se reentrena) y se reconstruye la mezcla de 0.60021. La celda verifica que los F1 coinciden con los de la sección 3.

In [ ]:
t0 = time.perf_counter()
base_runs = {s: XV.run_seed(X, y, s, CX.VARIANTS["base"], CX.BASE_NAMES, N_JOBS) for s in SEEDS}
p_base = {s: P.blend(r.probs, r.masks["mention"]) for s, r in base_runs.items()}
for s in SEEDS:
    print(f"Seed {s}: F1 mezcla vigente = {E.f1_macro(base_runs[s].y_idx, p_base[s]):.4f}")
np.testing.assert_allclose(p_base[AJ], pasos[AJ]["word_seq3"], atol=1e-12)
print(f"OK: coincide con la sección 3 ({time.perf_counter() - t0:.1f} s)")

## 8. Grupo A: preprocesamiento

Cada variante reentrena sus canales (con caché propia) y se mezcla **con los pesos vigentes**, para aislar el efecto del preprocesamiento.

- **TweetTokenizer**: `preserve_case=False`, `reduce_len=True`, `strip_handles=False`. Se tokeniza el texto original (no el ya pasado por casefold) para que `:D` siga siendo emoticono. Decodifica entidades HTML por sí mismo, así que A2 ya ve `<3`. `word_seq3` mantiene el mapeo URL / @USER / #TAG.
- **A4 / A5** solo quitan filas del **train** de cada semilla; validación y test se predicen completos.

Ejemplo de tokenización:

In [ ]:
from tweetage import preprocess as PP

for t in [
    "te amo &lt;3 te amooooo :D jajaja",
    "RT @Juan: Hola!!! mira esto... https://t.co/AbC #FelizDia",
    "(@ Starbucks) w/ @ana :(",
]:
    print(f"{t!r}\n  regex: {M.TOKEN_RE.findall(t.casefold())}\n  tweet: {list(PP.tweet_tokens(t))}\n")

Xtr_aj = splits[AJ][0]
print("Filas de train quitadas (semilla de ajuste):")
print(f"  A4 limpieza: {(~PP.clean_mask(Xtr_aj)).sum():,}")
print(f"  A5 plantillas ruidosas: {(~PP.template_filter_mask(Xtr_aj, splits[AJ][1])).sum():,}")

In [ ]:
A_VARIANTS = ["A1_unescape", "A2_tweettok", "A3_unescape_tweettok", "A4_clean", "A5_template"]
runs_A = {}
for v in A_VARIANTS:
    t0 = time.perf_counter()
    runs_A[v] = {s: XV.run_seed(X, y, s, CX.VARIANTS[v], CX.BASE_NAMES, N_JOBS) for s in SEEDS}
    cand = {s: P.blend(r.probs, r.masks["mention"]) for s, r in runs_A[v].items()}
    evaluar_v2(v, cand, {"grupo": "A", **CX.VARIANTS[v]}, "preprocesamiento; pesos vigentes")
    print(f"{v}: {time.perf_counter() - t0:.0f} s")
tabla("A")

## 9. Grupo B: mezcla (sin reentrenar)

- **B1**: 6 sesgos por clase sumados a log p, elegidos por búsqueda de coordenadas sobre F1 macro. Atacan el desbalance de predicciones (en test, 18-24 recibe 21 770 predicciones y 35-49 solo 15 434).
- **B2–B4**: en vez de los pesos greedy, se calibra la temperatura de cada canal (su propio log-loss) y luego se ajustan todos los pesos juntos minimizando el log-loss de la mezcla. Cada uno se evalúa con y sin sesgos.

Todo se ajusta en la semilla de ajuste; el Δ de ajuste es optimista por construcción, el que cuenta es el de confirmación.

In [ ]:
y_aj, y_cf = base_runs[AJ].y_idx, base_runs[CF].y_idx

bias_b1 = B.fit_bias(p_base[AJ], y_aj)
evaluar_v2(
    "B1_sesgos",
    {s: B.apply_bias(p_base[s], bias_b1) for s in SEEDS},
    {"grupo": "B", "bias": bias_b1.round(4).tolist()},
    "sesgos por clase sobre la mezcla vigente",
)
print("Sesgos:", dict(zip(CLASES, bias_b1.round(3))))
display(
    pd.DataFrame(
        {
            "real": E.pred_distribution(np.eye(6)[y_cf]),
            "vigente": E.pred_distribution(p_base[CF]),
            "con sesgos": E.pred_distribution(B.apply_bias(p_base[CF], bias_b1)),
        }
    ).T.rename_axis("predicciones (confirmación)")
)

In [ ]:
mezclas_B = {
    "B2_lineal": B.LinearBlend(CX.BASE_NAMES, CX.GATES_DEFAULT),
    "B3_lineal_compuertas": B.LinearBlend(CX.BASE_NAMES, CX.GATES_SOCIAL),
    "B4_loglineal": B.LogLinearBlend(CX.BASE_NAMES, CX.GATES_DEFAULT),
}
pesos_B = {}
for nombre, m in mezclas_B.items():
    m.fit(base_runs[AJ].probs, y_aj, base_runs[AJ].masks)
    cand = {s: m.predict_proba(base_runs[s].probs, base_runs[s].masks) for s in SEEDS}
    evaluar_v2(nombre, cand, {"grupo": "B", **m.to_dict()}, "mezcla ajustada en conjunto")
    bias = B.fit_bias(cand[AJ], y_aj)
    evaluar_v2(
        f"{nombre}+sesgos",
        {s: B.apply_bias(cand[s], bias) for s in SEEDS},
        {"grupo": "B", **m.to_dict(), "bias": bias.round(4).tolist()},
        "mezcla conjunta + sesgos",
    )
    pesos_B[nombre] = pd.Series(m.weights_)
    if hasattr(m, "temps_"):
        pesos_B[f"{nombre} (T)"] = pd.Series(m.temps_)
display(pd.DataFrame(pesos_B).round(3))
tabla("B")

## 10. Grupo C: canales nuevos

Se entrenan sobre el texto original (variante base) a temperatura 1. Cada canal se prueba como en la sección 4: grilla de temperatura × peso en la semilla de ajuste, confirmación en la otra.

- **C1** reemplaza a `word_seq3`: se agrega a la mezcla hasta `char_seq2`, igual que en la sección 4, y se compara contra la mezcla vigente completa.
- **C2–C5** se agregan sobre la mezcla vigente: (1 − peso) · vigente + peso · canal.
- **C6** no se mezcla en todas las filas: solo cuando 18-24 y 25-34 son las dos clases más probables, reparte la masa de ambas entre la proporción de la mezcla y la del especialista.

In [ ]:
t0 = time.perf_counter()
new_runs = {
    s: XV.run_seed(X, y, s, CX.VARIANTS["base"], CX.NEW_NAMES + CX.SPECIALISTS, N_JOBS) for s in SEEDS
}
print(f"Canales nuevos listos en {time.perf_counter() - t0:.0f} s")
pd.DataFrame(
    {
        canal: {
            f"f1_{col}": E.f1_macro(new_runs[s].y_idx, new_runs[s].probs[canal])
            for s, col in ((AJ, "ajuste"), (CF, "confirmacion"))
        }
        for canal in CX.NEW_NAMES
    }
).T.round(4).rename_axis("F1 del canal solo")

### 10.1 C1: Jelinek-Mercer en lugar de `word_seq3`

In [ ]:
grilla_c1 = B.grid_add(
    pasos[AJ]["char_seq2"],
    new_runs[AJ].probs["word_seq3_jm"],
    y_aj,
    temps=(0.1, 0.2, 0.3, 0.5, 1.0),
    weights=(0.10, 0.15, 0.20, 0.25, 0.30),
)
t_c1, w_c1 = float(grilla_c1.iloc[0]["temperatura"]), float(grilla_c1.iloc[0]["peso"])
cand = {s: B.add_channel(pasos[s]["char_seq2"], new_runs[s].probs["word_seq3_jm"], t_c1, w_c1) for s in SEEDS}
evaluar_v2(
    "C1_word_seq3_jm",
    cand,
    {"grupo": "C", "canal": "word_seq3_jm", "temperatura": t_c1, "peso": w_c1},
    "reemplaza word_seq3 (add-alpha) por Jelinek-Mercer",
)
print(f"C1: T={t_c1}, peso={w_c1}")

### 10.2 C2–C5: canales agregados sobre la mezcla vigente

In [ ]:
ADD = {
    "C2_char5_cs": "char5_cs",
    "C3_shape5": "shape5",
    "C3_shape7": "shape7",
    "C4_char_seq2_rev": "char_seq2_rev",
    "C4_char_seq4": "char_seq4",
    "C4_word_seq3_rev": "word_seq3_rev",
    "C5_length": "length",
}
config_add = {}
for nombre, canal in ADD.items():
    g = B.grid_add(p_base[AJ], new_runs[AJ].probs[canal], y_aj)
    t, w = float(g.iloc[0]["temperatura"]), float(g.iloc[0]["peso"])
    if w == 0:  # la grilla prefiere no agregarlo
        cand = p_base
    else:
        cand = {s: B.add_channel(p_base[s], new_runs[s].probs[canal], t, w) for s in SEEDS}
    config_add[nombre] = (t, w)
    evaluar_v2(
        nombre,
        cand,
        {"grupo": "C", "canal": canal, "temperatura": t, "peso": w},
        "canal agregado a la mezcla vigente",
    )
pd.DataFrame(config_add, index=["T", "peso"]).T

### 10.3 C6: especialistas 18-24 frente a 25-34

In [ ]:
config_spec = {}
for canal in CX.SPECIALISTS:
    g = B.grid_specialist(p_base[AJ], new_runs[AJ].probs[canal], y_aj)
    t, w = float(g.iloc[0]["temperatura"]), float(g.iloc[0]["peso"])
    cand = (
        p_base
        if w == 0
        else {s: B.specialist_apply(p_base[s], new_runs[s].probs[canal], t, w) for s in SEEDS}
    )
    config_spec[canal] = (t, w)
    evaluar_v2(
        f"C6_{canal}",
        cand,
        {"grupo": "C", "canal": canal, "temperatura": t, "peso": w},
        "especialista 18-24 vs 25-34",
    )
pd.DataFrame(config_spec, index=["T", "peso"]).T

### 10.4 Resultados del grupo C

In [ ]:
tabla("C")

## 11. Ranking de los grupos A–C

Ordenado por Δ en la semilla de confirmación. Ninguno pasa la regla por sí solo: las mejoras están en +0.0006 a +0.0011, dentro del ruido de una partición. Los que mejoran en ambas semillas (shape5/7, char_seq4, tokenizador) se combinan en las secciones 12 y 13.

In [ ]:
ranking = pd.DataFrame(resultados).set_index("experimento").sort_values("delta_confirmacion", ascending=False)
ranking[["grupo"] + COLS].round(4)

## 12. Combinación automática con mezcla conjunta (descartada)

Primer intento de combinar todo lo que mejoró. **Resultado: Δ ≈ 0 en confirmación**, porque las mezclas conjuntas (grupo B) optimizan log-loss y no F1 macro: aplanan los canales NB, que funcionan mejor "afilados". Se conserva como registro; la combinación que sí funciona está en la sección 13.

### 12.1 Reglas

Reglas fijadas **antes** de ver el resultado de la combinación:

1. **Texto**: entre A1, A2 y A3, la de mayor Δ medio que mejore en ambas semillas (si ninguna, texto original).
2. **Filas de train**: A4 y A5 se activan si cada una mejora en ambas semillas.
3. **Canales**: los 10 vigentes + los nuevos con Δ de confirmación > 0 (la mezcla conjunta les asigna peso; los inútiles quedan con peso ~0).
4. **Tipo de mezcla**: el de mayor Δ de confirmación entre B2, B3 y B4 (el Δ de ajuste es optimista porque ahí se ajustan los pesos).
5. **Especialista**: el de C6 que pase, con su temperatura y peso.
6. **Sesgos por clase** al final.

Todo se ajusta en la semilla de ajuste y se evalúa en la de confirmación.

In [ ]:
rk = pd.DataFrame(resultados).set_index("experimento")
mejora = lambda n: bool(rk.loc[n, "delta_ajuste"] > 0 and rk.loc[n, "delta_confirmacion"] > 0)

opciones_texto = [v for v in ("A1_unescape", "A2_tweettok", "A3_unescape_tweettok") if mejora(v)]
texto = max(opciones_texto, key=lambda v: rk.loc[v, "delta_medio"]) if opciones_texto else "base"
flags = {**CX.VARIANTS[texto], "clean": mejora("A4_clean"), "template": mejora("A5_template")}

nuevos = {"C1_word_seq3_jm": "word_seq3_jm", **ADD}
pool = CX.BASE_NAMES + [c for n, c in nuevos.items() if rk.loc[n, "delta_confirmacion"] > 0]
especialistas = [c for c in CX.SPECIALISTS if rk.loc[f"C6_{c}", "pasa"]]
especialista = (
    max(especialistas, key=lambda c: rk.loc[f"C6_{c}", "delta_confirmacion"]) if especialistas else None
)

# Por Δ de confirmación: el de ajuste es optimista porque ahí se ajustaron los pesos.
tipo = max(
    ("B2_lineal", "B3_lineal_compuertas", "B4_loglineal"), key=lambda n: rk.loc[n, "delta_confirmacion"]
)
crear_mezcla = {
    "B2_lineal": lambda names: B.LinearBlend(names, CX.GATES_DEFAULT),
    "B3_lineal_compuertas": lambda names: B.LinearBlend(names, CX.GATES_SOCIAL),
    "B4_loglineal": lambda names: B.LogLinearBlend(names, CX.GATES_DEFAULT),
}[tipo]

print("Variante:", CX.variant_name(flags), flags)
print("Canales:", pool)
print("Especialista:", especialista, "| mezcla:", tipo)

### 12.2 Resultado

In [ ]:
t0 = time.perf_counter()
final_runs = {
    s: XV.run_seed(X, y, s, flags, pool + ([especialista] if especialista else []), N_JOBS) for s in SEEDS
}
print(f"Canales de la combinación listos en {time.perf_counter() - t0:.0f} s")

mezcla_final = crear_mezcla(pool).fit(final_runs[AJ].probs, y_aj, final_runs[AJ].masks)
p_final = {s: mezcla_final.predict_proba(final_runs[s].probs, final_runs[s].masks) for s in SEEDS}
evaluar_v2(
    "FINAL_1_mezcla",
    p_final,
    {"grupo": "FINAL", "flags": flags, **mezcla_final.to_dict()},
    "combinación sin sesgos",
)

if especialista:
    t_sp, w_sp = config_spec[especialista]
    p_final = {
        s: B.specialist_apply(p_final[s], final_runs[s].probs[especialista], t_sp, w_sp) for s in SEEDS
    }
    evaluar_v2(
        "FINAL_2_especialista",
        p_final,
        {"grupo": "FINAL", "especialista": especialista},
        "combinación + especialista",
    )

bias_final = B.fit_bias(p_final[AJ], y_aj)
evaluar_v2(
    "FINAL_3_sesgos",
    {s: B.apply_bias(p_final[s], bias_final) for s in SEEDS},
    {"grupo": "FINAL", "bias": bias_final.round(4).tolist()},
    "combinación completa",
)
display(pd.Series(mezcla_final.weights_).sort_values(ascending=False).round(3).to_frame("peso"))
tabla("FINAL")

Resultado por clase y distribución de predicciones (confirmación):

In [ ]:
s_final = {s: B.apply_bias(p_final[s], bias_final) for s in SEEDS}
display(
    pd.DataFrame(
        {"vigente": E.f1_per_class(y_cf, p_base[CF]), "final v2": E.f1_per_class(y_cf, s_final[CF])}
    ).T.round(3)
)
display(
    pd.DataFrame(
        {
            "real": E.pred_distribution(np.eye(6)[y_cf]),
            "vigente": E.pred_distribution(p_base[CF]),
            "final v2": E.pred_distribution(s_final[CF]),
        }
    ).T
)

### 12.3 Especificaciones (solo como registro)

Se guardan tres specs de este intento. **Ninguna supera a la mezcla vigente**, así que no se recomiendan para Kaggle:

| Spec | Qué es | Δ confirmación |
|---|---|---|
| `final` | combinación automática completa | ≈ 0 |
| `mezcla_B` | los 10 canales vigentes con la mejor mezcla del grupo B | < 0 |
| `conservadora` | mezcla vigente + sesgos por clase (B1) | < 0 |

Los parámetros se reajustan con las validaciones de ambas semillas juntas.

In [ ]:
def resumen(nombre):
    r = rk_final.loc[nombre]
    return {
        "f1_confirmacion": float(r["f1_confirmacion"]),
        "delta_confirmacion": float(r["delta_confirmacion"]),
        "experimento": nombre,
    }


rk_final = pd.DataFrame(resultados).set_index("experimento")

# --- final: reajuste con ambas semillas
probs_ab, masks_ab, y_ab = XV.concat_runs([final_runs[AJ], final_runs[CF]])
m_ab = crear_mezcla(pool).fit(probs_ab, y_ab, masks_ab)
p_ab = m_ab.predict_proba(probs_ab, masks_ab)
spec_esp = None
if especialista:
    p_ab = B.specialist_apply(p_ab, probs_ab[especialista], t_sp, w_sp)
    spec_esp = {"name": especialista, "temperature": t_sp, "weight": w_sp}
spec_final = {
    "flags": flags,
    "channels": pool + ([especialista] if especialista else []),
    "blend": m_ab.to_dict(),
    "specialist": spec_esp,
    "bias": B.fit_bias(p_ab, y_ab).tolist(),
    **resumen("FINAL_3_sesgos"),
}
print(XV.save_spec("final", spec_final))

# --- mezcla_B: canales vigentes + mejor B (con o sin sesgos según confirmación)
probs_b, masks_b, y_b = XV.concat_runs([base_runs[AJ], base_runs[CF]])
m_b = {
    "B2_lineal": B.LinearBlend(CX.BASE_NAMES, CX.GATES_DEFAULT),
    "B3_lineal_compuertas": B.LinearBlend(CX.BASE_NAMES, CX.GATES_SOCIAL),
    "B4_loglineal": B.LogLinearBlend(CX.BASE_NAMES, CX.GATES_DEFAULT),
}[tipo].fit(probs_b, y_b, masks_b)
con_sesgos = rk_final.loc[f"{tipo}+sesgos", "delta_confirmacion"] > rk_final.loc[tipo, "delta_confirmacion"]
p_b = m_b.predict_proba(probs_b, masks_b)
spec_b = {
    "flags": CX.VARIANTS["base"],
    "channels": CX.BASE_NAMES,
    "blend": m_b.to_dict(),
    "specialist": None,
    "bias": B.fit_bias(p_b, y_b).tolist() if con_sesgos else None,
    **resumen(f"{tipo}+sesgos" if con_sesgos else tipo),
}
print(XV.save_spec("mezcla_B", spec_b))

# --- conservadora: mezcla vigente + sesgos
p_v = np.concatenate([p_base[AJ], p_base[CF]])
spec_c = {
    "flags": CX.VARIANTS["base"],
    "channels": CX.BASE_NAMES,
    "blend": {"kind": "pipeline_v1"},
    "specialist": None,
    "bias": B.fit_bias(p_v, y_b).tolist(),
    **resumen("B1_sesgos"),
}
print(XV.save_spec("conservadora", spec_c))

pd.DataFrame(
    {
        n: resumen(r)
        for n, r in (
            ("final", "FINAL_3_sesgos"),
            ("mezcla_B", spec_b["experimento"]),
            ("conservadora", "B1_sesgos"),
        )
    }
).T

## 13. Combinación greedy: reemplazos + canales agregados

**Hallazgo de la revisión de la tanda.** Los canales nuevos más fuertes por sí solos son `char_seq4` (F1 0.552) y `char5_cs` (0.551), por encima de `char5` (0.544). En la sección 10 solo se probaron como canales **agregados** con poco peso. Aquí se prueban como **reemplazos** dentro de la mezcla vigente, con el mismo peso y la misma posición:

- `charN` → `charN_cs` (caracteres **sin casefold**: recupera el patrón de mayúsculas en U del análisis exploratorio).
- Luego el tokenizador (A2) y, por último, `shape7` y `char_seq4` agregados con la grilla de la sección 10.

Cada paso se acepta solo si mejora **frente al paso anterior en ambas semillas**. Los reemplazos no tienen parámetros ajustados; solo los canales agregados eligen temperatura y peso, y lo hacen en la semilla de ajuste.

Se guardan dos especificaciones:

- `greedy`: el camino completo.
- `greedy_conservador`: solo reemplazos + tokenizador, sin ningún parámetro ajustado.

### 13.1 Reemplazos sin casefold (sin parámetros ajustados)

In [ ]:
CS = {"char4": "char4_cs", "char5": "char5_cs", "char6": "char6_cs"}
EXTRA_G = list(CS.values()) + ["shape7", "char_seq4"]
runs_g = {s: XV.run_seed(X, y, s, CX.VARIANTS["base"], CX.BASE_NAMES + EXTRA_G, N_JOBS) for s in SEEDS}
WORD_TT = ("word1", "position", "mention_pos", "word_seq3")


def probs_g(s, tokenizer):
    p = dict(runs_g[s].probs)
    if tokenizer:  # canales de palabras de la variante A2 (sección 8)
        p.update({k: runs_A["A2_tweettok"][s].probs[k] for k in WORD_TT})
    return p


def puntajes(spec):
    """Probabilidades de la mezcla (las specs G no llevan sesgos, así que el argmax es el mismo)."""
    tok = spec["flags"]["tweet_tok"]
    return {s: XV.predict_spec(spec, probs_g(s, tok), runs_g[s].masks, proba=True) for s in SEEDS}


def nueva_spec(swaps, tokenizer=False, adds=()):
    return {
        "flags": CX.VARIANTS["A2_tweettok" if tokenizer else "base"],
        "channels": list(CX.BASE_NAMES),
        "swaps": dict(swaps),
        "blend": {"kind": "pipeline_v1"},
        "adds": [dict(a) for a in adds],
        "specialist": None,
        "bias": None,
    }


def mejora_ambas(nuevo, anterior):
    return all(
        E.f1_macro(base_runs[s].y_idx, nuevo[s]) > E.f1_macro(base_runs[s].y_idx, anterior[s]) for s in SEEDS
    )


# Reemplazos sin casefold (sin parámetros)
caminos = {
    "G1_char5_cs": {"char5": "char5_cs"},
    "G1_char4_5_cs": {"char4": "char4_cs", "char5": "char5_cs"},
    "G1_char5_6_cs": {"char5": "char5_cs", "char6": "char6_cs"},
    "G1_char4_5_6_cs": dict(CS),
}
for nombre, swaps in caminos.items():
    evaluar_v2(nombre, puntajes(nueva_spec(swaps)), {"grupo": "G", "swaps": swaps}, "reemplazo sin casefold")
tabla("G")

### 13.2 Dos caminos: con y sin `char6_cs`

`char6_cs` mejora en ajuste pero no en confirmación por sí solo, así que se siguen dos caminos: el **conservador** (char4/5 sin casefold) y el **completo** (char4/5/6). Sobre cada uno se prueba el tokenizador y luego `shape7` y `char_seq4`; cada paso se acepta solo si mejora frente al anterior en ambas semillas.

In [ ]:
# Dos caminos: con y sin char6_cs
specs_g, filas_g = {}, []
for etiqueta, swaps in (("conservador", caminos["G1_char4_5_cs"]), ("completo", caminos["G1_char4_5_6_cs"])):
    spec = nueva_spec(swaps)
    actual = puntajes(spec)
    con_tok = nueva_spec(swaps, tokenizer=True)
    p_tok = puntajes(con_tok)
    evaluar_v2(f"G2_{etiqueta}_tokenizador", p_tok, {"grupo": "G", **con_tok}, "reemplazos + TweetTokenizer")
    if mejora_ambas(p_tok, actual):
        spec, actual = con_tok, p_tok
    for canal in ("shape7", "char_seq4"):
        g = B.grid_add(actual[AJ], runs_g[AJ].probs[canal], base_runs[AJ].y_idx)
        t, w = float(g.iloc[0]["temperatura"]), float(g.iloc[0]["peso"])
        if w == 0:
            continue
        cand_spec = {**spec, "adds": spec["adds"] + [{"name": canal, "temperature": t, "weight": w}]}
        cand = puntajes(cand_spec)
        evaluar_v2(f"G3_{etiqueta}_+{canal}", cand, {"grupo": "G", **cand_spec}, f"+ {canal} (T={t}, w={w})")
        acepta = mejora_ambas(cand, actual)
        filas_g.append({"camino": etiqueta, "canal": canal, "T": t, "peso": w, "aceptado": acepta})
        if acepta:
            spec, actual = cand_spec, cand
    specs_g[etiqueta] = (spec, actual)
display(pd.DataFrame(filas_g))
tabla("G")

### 13.3 Resultado de cada camino y especificaciones

In [ ]:
# Resultado de cada camino frente a la mezcla vigente y especificaciones para el 04
resumen_g = {}
for etiqueta, (spec, actual) in specs_g.items():
    nombre = f"GFINAL_{etiqueta}"
    fila = evaluar_v2(nombre, actual, {"grupo": "GFINAL", **spec}, "camino greedy completo")
    spec = {
        **spec,
        "experimento": nombre,
        "f1_confirmacion": fila["f1_confirmacion"],
        "delta_confirmacion": fila["delta_confirmacion"],
    }
    print(XV.save_spec("greedy" if etiqueta == "completo" else "greedy_conservador", spec))
    resumen_g[etiqueta] = {
        "swaps": spec["swaps"],
        "tokenizador": spec["flags"]["tweet_tok"],
        "agregados": [(a["name"], a["temperature"], a["weight"]) for a in spec["adds"]],
    }
display(pd.DataFrame(resumen_g).T)
display(
    pd.DataFrame(
        {
            "vigente": E.f1_per_class(y_cf, p_base[CF]),
            **{f"greedy {k}": E.f1_per_class(y_cf, v[1][CF]) for k, v in specs_g.items()},
        }
    ).T.round(3)
)
tabla("GFINAL")

## 14. Combinación con el LM de entidades (submission 0.60270 del equipo)

La submission de 0.60270 reemplaza `word_seq3` por un LM de palabras que conserva las menciones y hashtags que aparecen al menos 5 veces en train (las más raras se agrupan en `@USER` / `#TAG`). Ese cambio toca un canal distinto a los de esta sección, así que se prueba encima de los dos caminos greedy, con y sin TweetTokenizer. `word_seq3_ent5` reproduce el modelo del equipo dentro del paquete (`models_extra.EntityWordSeqLM`).

Se guardan dos especificaciones más: `greedy_ent` (camino completo + entidades) y `greedy_conservador_ent` (camino conservador + entidades).

In [ ]:
runs_e = {
    s: XV.run_seed(X, y, s, CX.VARIANTS["base"], ["word_seq3_ent5", "word_seq3_ent5_tt"], N_JOBS)
    for s in SEEDS
}


def puntajes_e(spec):
    out = {}
    for s in SEEDS:
        p = probs_g(s, spec["flags"]["tweet_tok"])
        p.update(runs_e[s].probs)
        out[s] = XV.predict_spec(spec, p, runs_g[s].masks, proba=True)
    return out


evaluar_v2(
    "E0_ent5",
    puntajes_e(nueva_spec({"word_seq3": "word_seq3_ent5"})),
    {"grupo": "E"},
    "LM de entidades del equipo (submission 0.60270)",
)
evaluar_v2(
    "E0_ent5_tt",
    puntajes_e(nueva_spec({"word_seq3": "word_seq3_ent5_tt"})),
    {"grupo": "E"},
    "LM de entidades con TweetTokenizer",
)
for etiqueta, nombre_spec in (("conservador", "greedy_conservador_ent"), ("completo", "greedy_ent")):
    base_spec = specs_g[etiqueta][0]
    for canal in ("word_seq3_ent5", "word_seq3_ent5_tt"):
        spec = {**base_spec, "swaps": {**base_spec["swaps"], "word_seq3": canal}}
        fila = evaluar_v2(
            f"E_{etiqueta}+{canal}", puntajes_e(spec), {"grupo": "E", **spec}, f"camino {etiqueta} + {canal}"
        )
    # se guarda la versión con tokenizador (la de mayor Δ en ambas semillas)
    print(
        XV.save_spec(
            nombre_spec,
            {
                **spec,
                "experimento": fila["experimento"],
                "f1_confirmacion": fila["f1_confirmacion"],
                "delta_confirmacion": fila["delta_confirmacion"],
            },
        )
    )
tabla("E")

## 15. Resumen global: experimentos, canales y Kaggle

Cinco tablas para el informe:

1. **Ranking global de experimentos**: originales (secciones 3 y 4) y nuevos (secciones 8–14, incluida la combinación con el LM de entidades del equipo). Ojo con la base de comparación: los originales se miden frente al **paso anterior** de la mezcla incremental (o frente a la mezcla hasta `char_seq2` en la sección 4), y los nuevos frente a la **mezcla vigente** (0.60021).
2. **Ranking de canales**: F1 de cada canal por sí solo, en ambas semillas.
3. **Canales similares**: cada canal nuevo junto a su versión original y el efecto que tuvo dentro de la mezcla.
4. **Experimentos similares**: experimentos originales y nuevos agrupados por la idea que prueban.
5. **Validación local frente a Kaggle**: todas las submissions con score público conocido.

### 15.1 Ranking global de experimentos

In [ ]:
KAGGLE_S4 = {
    "lm_palabras_orden2 (T=0.3, peso=0.15)": 0.59647,
    "lm_palabras_orden3 (T=0.2, peso=0.2)": 0.60021,
}
filas = []
for paso, r in tabla_mezcla.iterrows():  # sección 3: mezcla incremental
    if pd.isna(r["delta_ajuste"]):
        continue
    filas.append(
        {
            "experimento": paso,
            "origen": "original",
            "grupo": "mezcla incremental",
            "comparado_contra": "paso anterior",
            "f1_confirmacion": r["f1_confirmacion"],
            "delta_ajuste": r["delta_ajuste"],
            "delta_confirmacion": r["delta_confirmacion"],
            "kaggle": r["kaggle"],
        }
    )
base_s4 = confirmacion.loc["base (hasta char_seq2)"]
for variante, r in confirmacion.drop("base (hasta char_seq2)").iterrows():  # sección 4
    filas.append(
        {
            "experimento": variante,
            "origen": "original",
            "grupo": "orden LM palabras",
            "comparado_contra": "mezcla hasta char_seq2",
            "f1_confirmacion": r["f1_confirmacion"],
            "delta_ajuste": r["f1_ajuste"] - base_s4["f1_ajuste"],
            "delta_confirmacion": r["f1_confirmacion"] - base_s4["f1_confirmacion"],
            "kaggle": KAGGLE_S4.get(variante, np.nan),
        }
    )
for r in resultados:  # secciones 8–14
    filas.append(
        {
            "experimento": r["experimento"],
            "origen": "v2",
            "grupo": r["grupo"],
            "comparado_contra": "mezcla vigente",
            "f1_confirmacion": r["f1_confirmacion"],
            "delta_ajuste": r["delta_ajuste"],
            "delta_confirmacion": r["delta_confirmacion"],
            "kaggle": KAGGLE_V2.get(r["experimento"], np.nan),
            "ic95_bajo": r["ic95_bajo"],
            "pasa": r["pasa"],
        }
    )
ranking_global = (
    pd.DataFrame(filas).set_index("experimento").sort_values("delta_confirmacion", ascending=False)
)
ranking_global.round(4)

### 15.2 Ranking de canales (F1 de cada canal por sí solo)

In [ ]:
def f1_canal(p, s):
    return E.f1_macro(E.to_idx(splits[s][3]), p)


fuentes = {}  # canal -> {semilla: probabilidades}
for nombre in [*P.CHANNELS, "word_seq2_t1"]:
    fuentes[nombre.replace("_t1", "")] = {s: probs[s][nombre] for s in SEEDS}
for nombre in WORD_TT:
    fuentes[f"{nombre}_tt"] = {s: runs_A["A2_tweettok"][s].probs[nombre] for s in SEEDS}
for nombre in CX.NEW_NAMES:
    fuentes[nombre] = {s: new_runs[s].probs[nombre] for s in SEEDS}
for nombre in EXTRA_G:
    fuentes[nombre] = {s: runs_g[s].probs[nombre] for s in SEEDS}
for nombre in ("word_seq3_ent5", "word_seq3_ent5_tt"):
    fuentes[nombre] = {s: runs_e[s].probs[nombre] for s in SEEDS}

ORIGINALES = set(P.CHANNELS) | {"word_seq2", "word_seq3_ent5"}  # ent5: submission 0.60270 del equipo
ranking_canales = pd.DataFrame(
    [
        {
            "canal": n,
            "origen": "original" if n in ORIGINALES else "nuevo",
            "f1_ajuste": f1_canal(p[AJ], AJ),
            "f1_confirmacion": f1_canal(p[CF], CF),
        }
        for n, p in fuentes.items()
    ]
).set_index("canal")
ranking_canales["f1_medio"] = ranking_canales[["f1_ajuste", "f1_confirmacion"]].mean(axis=1)
ranking_canales.sort_values("f1_medio", ascending=False).round(4)

### 15.3 Canales similares: versión original frente a versiones nuevas

In [ ]:
FAMILIAS = {
    "caracteres 4": ["char4", "char4_cs"],
    "caracteres 5": ["char5", "char5_cs"],
    "caracteres 6": ["char6", "char6_cs"],
    "palabras (unigramas)": ["word1", "word1_tt"],
    "posición de palabras": ["position", "position_tt"],
    "menciones con posición": ["mention_pos", "mention_pos_tt"],
    "LM secuencial de palabras": [
        "word_seq2",
        "word_seq3",
        "word_seq3_ent5",
        "word_seq3_ent5_tt",
        "word_seq3_tt",
        "word_seq3_jm",
        "word_seq3_rev",
    ],
    "LM secuencial de caracteres": ["char_seq2", "char_seq2_rev", "char_seq4"],
    "forma del texto": ["shape5", "shape7"],
    "longitud": ["length"],
}
# Experimento donde se midió el efecto del canal dentro de la mezcla (Δ de confirmación)
EFECTO = {
    "char5_cs": "G1_char5_cs",
    "char4_cs": "G1_char4_5_cs",
    "char6_cs": "G1_char5_6_cs",
    "word1_tt": "A2_tweettok",
    "position_tt": "A2_tweettok",
    "mention_pos_tt": "A2_tweettok",
    "word_seq3_tt": "A2_tweettok",
    "word_seq3_jm": "C1_word_seq3_jm",
    "word_seq3_ent5": "E0_ent5",
    "word_seq3_ent5_tt": "E0_ent5_tt",
    "word_seq3_rev": "C4_word_seq3_rev",
    "char_seq2_rev": "C4_char_seq2_rev",
    "char_seq4": "C4_char_seq4",
    "shape5": "C3_shape5",
    "shape7": "C3_shape7",
    "length": "C5_length",
    "position": "mezcla_4_position",
    "mention_pos": "mezcla_5_mention_pos",
    "char_seq2": "mezcla_6_char_seq2",
    "word_seq2": "lm_palabras_orden2 (T=0.3, peso=0.15)",
    "word_seq3": "lm_palabras_orden3 (T=0.2, peso=0.2)",
}
filas = []
for familia, canales in FAMILIAS.items():
    ref = canales[0]
    for c in canales:
        exp = EFECTO.get(c)
        filas.append(
            {
                "familia": familia,
                "canal": c,
                "origen": "original" if c in ORIGINALES else "nuevo",
                "f1_confirmacion": ranking_canales.loc[c, "f1_confirmacion"],
                "delta_f1_vs_primero": ranking_canales.loc[c, "f1_confirmacion"]
                - ranking_canales.loc[ref, "f1_confirmacion"],
                "experimento_en_mezcla": exp or "parte de char_mix",
                "delta_mezcla_confirmacion": ranking_global.loc[exp, "delta_confirmacion"]
                if exp in ranking_global.index
                else np.nan,
            }
        )
pd.DataFrame(filas).set_index(["familia", "canal"]).round(4)

En la tabla anterior, `delta_f1_vs_primero` compara cada canal con el primero de su familia (la versión original) por sí solo. `delta_mezcla_confirmacion` es el efecto dentro de la mezcla en la semilla de confirmación: para los canales originales es el Δ del paso en que se agregaron (frente al paso anterior); para los nuevos, el Δ frente a la mezcla vigente del experimento indicado. Un canal puede ser mejor por sí solo y no aportar a la mezcla (o al revés) si repite información que otro canal ya aporta.

### 15.4 Experimentos similares, agrupados por la idea que prueban

In [ ]:
FAMILIAS_EXP = {
    "agregar un LM secuencial": [
        "mezcla_6_char_seq2",
        "lm_palabras_orden2 (T=0.3, peso=0.15)",
        "lm_palabras_orden3 (T=0.2, peso=0.2)",
        "C4_char_seq4",
        "C4_char_seq2_rev",
        "C4_word_seq3_rev",
        "G3_completo_+char_seq4",
    ],
    "mejorar el LM de palabras": [
        "C1_word_seq3_jm",
        "A2_tweettok",
        "A3_unescape_tweettok",
        "E0_ent5",
        "E0_ent5_tt",
    ],
    "canales de estilo y forma": [
        "C2_char5_cs",
        "C3_shape5",
        "C3_shape7",
        "C5_length",
        "G3_completo_+shape7",
    ],
    "caracteres sin casefold (reemplazo)": [
        "G1_char5_cs",
        "G1_char4_5_cs",
        "G1_char5_6_cs",
        "G1_char4_5_6_cs",
    ],
    "menciones, hashtags y posición": [
        "mezcla_2_hashtags",
        "mezcla_3_mentions",
        "mezcla_4_position",
        "mezcla_5_mention_pos",
        "B3_lineal_compuertas",
    ],
    "preprocesamiento y limpieza": ["A1_unescape", "A4_clean", "A5_template"],
    "pesos y sesgos de la mezcla": [
        "B1_sesgos",
        "B2_lineal",
        "B2_lineal+sesgos",
        "B3_lineal_compuertas+sesgos",
        "B4_loglineal",
        "B4_loglineal+sesgos",
    ],
    "especialistas 18-24 vs 25-34": ["C6_spec_char5", "C6_spec_word1"],
    "combinaciones": [
        "FINAL_1_mezcla",
        "FINAL_3_sesgos",
        "GFINAL_conservador",
        "GFINAL_completo",
        "E_conservador+word_seq3_ent5_tt",
        "E_completo+word_seq3_ent5_tt",
    ],
}
filas = []
for familia, exps in FAMILIAS_EXP.items():
    for e in exps:
        if e not in ranking_global.index:
            continue
        r = ranking_global.loc[e]
        filas.append(
            {
                "familia": familia,
                "experimento": e,
                "origen": r["origen"],
                "comparado_contra": r["comparado_contra"],
                "delta_ajuste": r["delta_ajuste"],
                "delta_confirmacion": r["delta_confirmacion"],
                "kaggle": r["kaggle"],
            }
        )
tabla_familias = pd.DataFrame(filas).set_index(["familia", "experimento"])
display(tabla_familias.round(4))

resumen_familias = (
    tabla_familias.reset_index()
    .groupby("familia")
    .agg(
        experimentos=("experimento", "size"),
        mejor=("delta_confirmacion", "max"),
        mejor_experimento=(
            "delta_confirmacion",
            lambda s: tabla_familias.reset_index().loc[s.idxmax(), "experimento"],
        ),
        mejoran_ambas=(
            "delta_confirmacion",
            lambda s: int(((s > 0) & (tabla_familias.reset_index().loc[s.index, "delta_ajuste"] > 0)).sum()),
        ),
    )
    .sort_values("mejor", ascending=False)
)
resumen_familias.round(4)

### 15.5 Validación local frente a Kaggle

Todas las submissions con score público conocido, en el orden en que se enviaron. Cada Δ se calcula frente a la submission anterior, tanto en la semilla de confirmación como en Kaggle. La razón Kaggle / local indica cuánto de la mejora local se trasladó al leaderboard público.

In [ ]:
KAGGLE_HIST = [
    ("+ position", "mezcla_4_position", 0.58850),
    ("+ mention_pos + char_seq2", "mezcla_6_char_seq2", 0.59281),
    ("+ LM de palabras orden 2", "lm_palabras_orden2 (T=0.3, peso=0.15)", 0.59647),
    ("LM de palabras orden 3 (vigente)", "lm_palabras_orden3 (T=0.2, peso=0.2)", 0.60021),
    ("LM de entidades ent5 (equipo)", "E0_ent5", 0.60270),
]
tabla_kaggle = pd.DataFrame(
    [
        {
            "submission": nombre,
            "experimento": exp,
            "f1_confirmacion": ranking_global.loc[exp, "f1_confirmacion"],
            "kaggle_publico": k,
        }
        for nombre, exp, k in KAGGLE_HIST
    ]
).set_index("submission")
tabla_kaggle["delta_local"] = tabla_kaggle["f1_confirmacion"].diff()
tabla_kaggle["delta_kaggle"] = tabla_kaggle["kaggle_publico"].diff()
tabla_kaggle["razon_kaggle_local"] = tabla_kaggle["delta_kaggle"] / tabla_kaggle["delta_local"]
tabla_kaggle.round(4)

En las cuatro transiciones, la mejora en Kaggle fue entre 0.86 y 1.16 veces la mejora en la semilla de confirmación. La validación con dos semillas y particiones sin textos compartidos es un buen predictor del leaderboard público para cambios de este tamaño.

## 16. Conclusiones

**Qué mejoró** (Δ en la semilla de confirmación frente a la mezcla vigente de 0.60021):

| Cambio | Δ confirmación | Por qué funciona |
|---|---|---|
| Caracteres sin casefold en char4 y char5 (reemplazo) | +0.0026 | recupera el uso de mayúsculas, que tiene forma de U por edad (análisis exploratorio) |
| LM de entidades (umbral 5, submission del equipo) | +0.0021 (Kaggle +0.0025) | conserva las cuentas y hashtags frecuentes, que son una señal social fuerte |
| TweetTokenizer en los canales de palabras | +0.0007 a +0.0018 (incremental) | emoticonos y `<3` completos, alargamientos unificados, `...` como token |
| `shape7` y `char_seq4` agregados (camino completo) | +0.0016 y +0.0015 (incrementales) | estilo independiente del vocabulario y contexto de 4 caracteres |

**Qué no mejoró:** sesgos por clase (sobreajuste a una partición), mezclas conjuntas que optimizan log-loss en vez de F1, filtrar plantillas ruidosas (quita ejemplos con señal), interpolación Jelinek-Mercer, Markov en reversa, longitud como evento y especialistas 18-24 / 25-34.

**Candidatas para Kaggle** (specs guardadas para `04_submission_v2.ipynb`):

| Spec | Qué combina | Δ confirmación | IC 95 % |
|---|---|---|---|
| `greedy_ent` | char4/5/6 sin casefold + tokenizador + shape7 + char_seq4 + entidades | +0.0068 | [+0.0042, +0.0097] |
| `greedy_conservador_ent` | char4/5 sin casefold + tokenizador + entidades | +0.0060 | [+0.0038, +0.0082] |
| `greedy` | camino completo sin entidades | +0.0051 | [+0.0026, +0.0079] |
| `greedy_conservador` | camino conservador sin entidades | +0.0044 | [+0.0025, +0.0063] |

**Limitaciones:**

- La validación usa dos semillas (un fold de 5 cada una), no validación cruzada completa.
- En el camino completo, aceptar `shape7` y `char_seq4` miró ambas semillas, así que su Δ es algo optimista. Las specs conservadoras no ajustan ningún parámetro y su Δ es la estimación más limpia.
- Las clases centrales (18-24 y 25-34) siguen siendo las más difíciles (F1 ≈ 0.50 y 0.48): sus estilos se parecen entre sí y a las clases vecinas.